# Notebook 2: Gradient Rank and Principal-angle Dynamics

This notebook reads the saved JAX experiment logs and visualizes gradient dynamics. It does not recompute training.

For each tracked weight matrix, the experiment logs:

| Matrix kind | Meaning |
|---|---|
| `gradient` | raw loss gradient for the weight matrix |
| `momentum` | AdamW first-moment buffer after gradient clipping |
| `update` | actual parameter update applied by AdamW |
| `activation` | linear-layer fan-in used in `grad_W = residual.T @ activation` |
| `residual` | backpropagated linear-output residual |

Each quantity has two rank plots: stable rank and 90% PCA rank. Principal-angle plots measure adjacent-step dominant-direction drift.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

import subprocess
import pandas as pd
from IPython.display import display

from clean_jax_exp.visualize import (
    latest_run,
    plot_matrix_metric,
    plot_angle_metric,
)

RUN_TRAINING = False
SAVE_PDF = False
OUTPUT_ROOT = Path("results/clean_jax_gradient")

GRADIENT_CMD = [
    "/Users/tongtongliang/miniforge3/bin/python3.12",
    "run_gradient_analysis_experiment.py",
    "--output-root", str(OUTPUT_ROOT),
    "--ambient-dim", "512",
    "--n-samples", "8192",
    "--width", "256",
    "--depth", "5",
    "--time-embed-dim", "256",
    "--steps", "2000",
    "--batch-size", "256",
    "--metric-every", "20",
    "--print-every", "50",
    "--lr", "1e-4",
    "--grad-clip-norm", "1.0",
]

if RUN_TRAINING:
    subprocess.run(GRADIENT_CMD, check=True)

RUN_DIR = latest_run(OUTPUT_ROOT / "runs")
print(f"Using run: {RUN_DIR}")

LAYERS_TO_SHOW = ["input_proj", "block5_mlp0", "output_proj"]

## Sanity Check

The key local factorization is checked numerically during training:

`grad_W = residual.T @ activation`

Small relative error means the activation and residual matrices are aligned with the actual autograd gradient.

In [ ]:
sanity = pd.read_csv(RUN_DIR / "logs" / "sanity_metrics.csv")
display(sanity.groupby(["mode", "layer"])["relative_error"].max().unstack().round(8))
print("max relative error", sanity["relative_error"].max())

## Rank Dynamics

Each plot compares `x`, `v`, and `eps` for one matrix kind and one layer. The legend is local to each plot, and no plot title is embedded in the image.

In [ ]:
for layer in LAYERS_TO_SHOW:
    for matrix_kind in ["gradient", "momentum", "update", "activation", "residual"]:
        plot_matrix_metric(RUN_DIR, matrix_kind=matrix_kind, layer=layer, metric="stable_rank", save_pdf=SAVE_PDF)
        plot_matrix_metric(RUN_DIR, matrix_kind=matrix_kind, layer=layer, metric="rank90", save_pdf=SAVE_PDF)

## Principal-angle Dynamics

These plots help test whether apparently low-rank gradients are stable directions or rotating low-rank directions. Rotating low-rank gradients can accumulate into high-rank momentum or updates.

In [ ]:
for layer in LAYERS_TO_SHOW:
    for angle_kind in ["adjacent_gradient", "gradient_vs_previous_momentum", "adjacent_activation", "adjacent_residual"]:
        plot_angle_metric(RUN_DIR, angle_kind=angle_kind, layer=layer, save_pdf=SAVE_PDF)

## Compact Tables

A quick numerical summary of the rank and angle logs.

In [ ]:
matrix_df = pd.read_csv(RUN_DIR / "logs" / "matrix_metrics.csv")
angle_df = pd.read_csv(RUN_DIR / "logs" / "angle_metrics.csv")

display(matrix_df.groupby(["matrix_kind", "layer", "mode"])[["stable_rank", "rank90"]].mean().round(2))
display(angle_df.groupby(["angle_kind", "layer", "mode"])["angle_deg"].mean().round(2))